In [1]:
# Importando a biblioteca pandas
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

# Carrega as credenciais invisíveis
load_dotenv()

# Cria o motor de conexão
engine = create_engine(f"postgresql+psycopg2://{os.getenv('DB_USER')}:{os.getenv('DB_PASS')}@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT')}/{os.getenv('DB_NAME')}")

# A query SQL para puxar os dados
query_extracao = "SELECT * FROM supply_chain_logistics"

# A mágica da ingestão: O Python executa a query e joga no DataFrame
df_logistica = pd.read_sql(query_extracao, engine)

# Exibe as primeiras 5 linhas para confirmar
display(df_logistica.head())


,id_pedido,data_pedido,origem,destino_estado,transportadora,valor_produto,tipo_frete,previsao_entrega,data_entrega_real,status_entrega,custo_frete
0,50001,2025-06-10 14:37:49.158969,Armazém MG,PR,Logística Express,1684.78,Expresso,2025-06-13 14:37:49.158969,2025-06-13 14:37:49.158969,Entregue,336.96
1,50002,2025-09-22 14:37:49.158969,Centro de Distribuição SP,RS,Transportes Nacionais,107.90,Padrão,2025-09-30 14:37:49.158969,2025-10-05 14:37:49.158969,Entregue com Atraso,10.79
2,50003,2025-10-02 14:37:49.158969,Centro de Distribuição SP,SP,TransGlobal,2171.06,Padrão,2025-10-10 14:37:49.158969,2025-10-15 14:37:49.158969,Entregue com Atraso,217.11
3,50004,2025-07-23 14:37:49.158969,Fábrica SC,PR,Logística Express,874.99,Padrão,2025-07-31 14:37:49.158969,2025-07-31 14:37:49.158969,Entregue,87.50
4,50005,2025-03-15 14:37:49.158969,Centro de Distribuição SP,DF,TransGlobal,1936.88,Expresso,2025-03-18 14:37:49.158969,2025-03-18 14:37:49.158969,Entregue,387.38


In [ ]:
# Informações gerais do DataFrame
df_logistica.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   id_pedido          8000 non-null   int64         
 1   data_pedido        8000 non-null   datetime64[ns]
 2   origem             8000 non-null   object        
 3   destino_estado     8000 non-null   object        
 4   transportadora     8000 non-null   object        
 5   valor_produto      8000 non-null   float64       
 6   tipo_frete         8000 non-null   object        
 7   previsao_entrega   8000 non-null   datetime64[ns]
 8   data_entrega_real  7792 non-null   datetime64[ns]
 9   status_entrega     8000 non-null   object        
 10  custo_frete        8000 non-null   float64       
dtypes: datetime64[ns](3), float64(2), int64(1), object(5)
memory usage: 687.6+ KB


In [5]:
df_logistica.describe(include='all')

,id_pedido,data_pedido,origem,destino_estado,transportadora,valor_produto,tipo_frete,previsao_entrega,data_entrega_real,status_entrega,custo_frete
count,8000.00000,8000,8000,8000,8000,8000.000000,8000,8000,7792,8000,8000.000000
unique,NaN,NaN,3,10,4,NaN,2,NaN,NaN,3,NaN
top,NaN,NaN,Centro de Distribuição SP,SP,Logística Express,NaN,Padrão,NaN,NaN,Entregue,NaN
freq,NaN,NaN,4024,856,2074,NaN,5975,NaN,NaN,5453,NaN
mean,54000.50000,2025-09-12 06:35:46.769609728,NaN,NaN,NaN,1266.859335,NaN,2025-09-19 00:13:16.769609728,2025-09-19 05:56:06.910886912,NaN,159.015084
min,50001.00000,2025-03-15 14:37:49.158969,NaN,NaN,NaN,50.750000,NaN,2025-03-18 14:37:49.158969,2025-03-16 14:37:49.173998,NaN,5.090000
25%,52000.75000,2025-06-13 14:37:49.173998080,NaN,NaN,NaN,660.530000,NaN,2025-06-20 14:37:49.173998080,2025-06-20 14:37:49.173998080,NaN,75.940000
50%,54000.50000,2025-09-12 14:37:49.158969088,NaN,NaN,NaN,1263.285000,NaN,2025-09-18 14:37:49.173998080,2025-09-19 14:37:49.158969088,NaN,143.780000
75%,56000.25000,2025-12-11 14:37:49.158969088,NaN,NaN,NaN,1868.502500,NaN,2025-12-17 14:37:49.173998080,2025-12-17 14:37:49.173998080,NaN,215.912500
max,58000.00000,2026-03-14 14:37:49.173998,NaN,NaN,NaN,2499.960000,NaN,2026-03-22 14:37:49.173998,2026-03-31 14:37:49.173998,NaN,499.740000


In [6]:
# Trazendo a quantidade de valores ausentes por coluna
df_logistica.isnull().sum()

id_pedido              0
data_pedido            0
origem                 0
destino_estado         0
transportadora         0
valor_produto          0
tipo_frete             0
previsao_entrega       0
data_entrega_real    208
status_entrega         0
custo_frete            0
dtype: int64

In [7]:
# 1. Calculando o tempo de trânsito em dias (Data Entrega - Data Pedido)
# O .dt.days extrai apenas o número inteiro de dias. Para os extraviados, continuará como NaN.
df_logistica['tempo_transito_dias'] = (df_logistica['data_entrega_real'] - df_logistica['data_pedido']).dt.days

# 2. Calculando a quantidade de dias de atraso
# Se entregou antes ou no prazo, o atraso é 0. Se atrasou, mostra quantos dias. Extraviados ficam nulos.
df_logistica['dias_atraso'] = (df_logistica['data_entrega_real'] - df_logistica['previsao_entrega']).dt.days
df_logistica['dias_atraso'] = df_logistica['dias_atraso'].apply(lambda x: x if x > 0 else 0)

# 3. Validando os dados (Você vai ver que os nulos continuam lá, intactos e corretos)
print("--- Validação de Nulos ---")
print(df_logistica[['status_entrega', 'tempo_transito_dias', 'dias_atraso']].isna().sum())

# Mostrando algumas linhas onde houve extravio para confirmar
display(df_logistica[df_logistica['status_entrega'] == 'Extraviado/Danificado'].head())

--- Validação de Nulos ---
status_entrega           0
tempo_transito_dias    208
dias_atraso              0
dtype: int64


,id_pedido,data_pedido,origem,destino_estado,transportadora,valor_produto,tipo_frete,previsao_entrega,data_entrega_real,status_entrega,custo_frete,tempo_transito_dias,dias_atraso
96,50097,2025-06-04 14:37:49.158969,Centro de Distribuição SP,PR,Logística Express,1684.03,Padrão,2025-06-12 14:37:49.158969,NaT,Extraviado/Danificado,168.40,NaN,0.0
116,50117,2026-02-24 14:37:49.158969,Armazém MG,BA,Logística Express,2355.45,Padrão,2026-03-04 14:37:49.158969,NaT,Extraviado/Danificado,235.54,NaN,0.0
146,50147,2025-11-15 14:37:49.158969,Fábrica SC,DF,TransGlobal,1288.09,Expresso,2025-11-18 14:37:49.158969,NaT,Extraviado/Danificado,257.62,NaN,0.0
187,50188,2025-10-04 14:37:49.158969,Armazém MG,RJ,Transportes Nacionais,352.07,Padrão,2025-10-12 14:37:49.158969,NaT,Extraviado/Danificado,35.21,NaN,0.0
238,50239,2026-01-01 14:37:49.158969,Centro de Distribuição SP,SC,Carga Rápida S.A.,446.62,Expresso,2026-01-04 14:37:49.158969,NaT,Extraviado/Danificado,89.32,NaN,0.0
